# 00_Initial_Raw_Data_Sorting

SSA Developed Code for Processing of Flow Crystallisation PXRD data attained From Beamline I11 at DLS

This notebook performs **initial organisiation of raw beamline data** prior to any diffraction processing and sorting

It:
- removes unused PXRD processing folders 
- organises auxilary files (diode plots, videos)
- groups Diffraction files '.nxs' and '.hdf' files into user-defined run folders based on collection number ranges 


## Workflow modes

This notebook supports two execution modes:

### Initial cleanup (run once per beamtime)
- Moves unused PXRD folders
- Sorts auxiliary `.dat` and `.avi` files
- Creates standard top-level folders

### Run definition (run multiple times)
- Creates new run folders
- Organises `.nxs` and `.hdf` files by collection number
- Does **not** reprocess auxiliary files

Use the toggle below to control the workflow.


In [ ]:
# Cell 1 : Imports and User Inputs 

# Please define: 
# - base directory containing raw beamline files 
# - a descriptive run name
# - the start and end collection number for this run. 

from pathlib import Path 
import shutil 


# set to true only once per beamtime for organisation of diode plots and avi files 
run_initial_cleanup = False #<-- set to True only once per beamtime

# User inputs
# path to raw beamline directory 
base_dir = Path(r"D:/00_sorting_script_test/RAW") # <-- please set


# descriptive run name
run_name = "Run_1"  # <-- please set

# start and end collection number for this run
start_run = 107537   # <-- please set
end_run = 107544    # <-- please set



In [ ]:
# Cell 2: Folder Definitions 

unused_folders = ['processed', 'processing', 'spool', 'tmp', 'xml']

diode_folder = base_dir / "diode_plots"
video_folder = base_dir / "videos"
unused_outputs = base_dir / "unused_beamline_outputs"

run_folder = base_dir / run_name 

In [ ]:
# Cell 3: Create required folder 

# always required 
run_folder.mkdir(exist_ok=True)

#only required in initial cleanup
if run_initial_cleanup:
    for folder in [diode_folder, video_folder, unused_outputs]:
        folder.mkdir(exist_ok=True)
print ("Folder Set Up Complete")


In [ ]:
# Cell 4 : Move unused beamline output folders
if run_initial_cleanup:
    for name in unused_folders:
        src = base_dir / name
        if src.exists() and src.is_dir():
            dest = unused_outputs / name
            shutil.move(str(src), str(dest))
            print(f"Moved folder {name} to unused outputs.")

In [ ]:
# Cell 5 : Sort Diode plots and avi files 

if run_initial_cleanup:
    for file in base_dir.iterdir():
        if file.is_file():
            if file.suffix == ".dat":
                shutil.move(str(file), str(diode_folder / file.name))
            elif file.suffix == ".avi":
                shutil.move(str(file), str(video_folder / file.name))
    print ("Diode plots and avi files moved.")

## Orginisation of Raw Diffraction Data 
the following step groups '.nxs' and '.hdf' files into new run folder based off of specified collection number range 

this step can be executed multiple times without re-running initial cleanup...

In [ ]:
# Cell 6: Organise Run Data 

missing_nxs = []
missing_hdf = []

for run in range(start_run, end_run + 1):
    nxs_file = base_dir / f"i11-1-{run}.nxs"
    hdf_file = base_dir / f"pixium_{run}.hdf"

    if nxs_file.exists():
        shutil.move(str(nxs_file), str(run_folder / nxs_file.name))
    else:
        missing_nxs.append(run)
    
    if hdf_file.exists():
        shutil.move(str(hdf_file), str(run_folder / hdf_file.name))
    else:
        missing_hdf.append(run)

print("Data Organisation Complete.")


In [ ]:
# Cell 7 - Summary of Missing Files
if missing_nxs:
    print("Missing .nxs files for runs:", missing_nxs)